# 07 - Incremental Load Pattern

## Purpose

Show a practical beginner-friendly pattern for incremental transaction ingestion in Microsoft Fabric.

This notebook is intentionally written as a pattern, not a full production CDC framework. It teaches the core idea: keep a watermark, read only new source rows, merge them into a Delta target, and record the new watermark.

## When to Use

Use this pattern when source data grows over time and full reloads become slow, expensive, or operationally risky.

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

source_path = "Files/retail_banking/source/transactions.csv"
bronze_table_name = "bronze_transactions_incremental"
watermark_table_name = "audit_watermark"
batch_id = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

print(f"Incremental batch ID: {batch_id}")

In [ ]:
# Create a simple watermark table if it does not exist.
# In production, include source system, entity, load type, and owner metadata.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {watermark_table_name} (
    entity_name STRING,
    watermark_column STRING,
    watermark_value STRING,
    updated_at TIMESTAMP
)
USING DELTA
""")

watermark_rows = (
    spark.table(watermark_table_name)
    .filter((F.col("entity_name") == "transactions") & (F.col("watermark_column") == "transaction_timestamp"))
    .orderBy(F.col("updated_at").desc())
    .limit(1)
    .collect()
)

last_watermark = watermark_rows[0]["watermark_value"] if watermark_rows else "1900-01-01 00:00:00"
print(f"Last processed watermark: {last_watermark}")

In [ ]:
# Read source transactions and filter to rows newer than the watermark.
source_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(source_path)
    .withColumn("transaction_timestamp", F.to_timestamp("transaction_timestamp"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file_name", F.input_file_name())
    .withColumn("_batch_id", F.lit(batch_id))
)

incremental_df = source_df.filter(F.col("transaction_timestamp") > F.to_timestamp(F.lit(last_watermark)))
incremental_count = incremental_df.count()
print(f"Rows selected for incremental load: {incremental_count}")
display(incremental_df.orderBy("transaction_timestamp"))

In [ ]:
# Merge into a Bronze incremental table.
# The merge makes the load idempotent by transaction_id.
if not spark.catalog.tableExists(bronze_table_name):
    incremental_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table_name)
    print(f"Created {bronze_table_name}")
elif incremental_count > 0:
    target = DeltaTable.forName(spark, bronze_table_name)
    (
        target.alias("target")
        .merge(incremental_df.alias("source"), "target.transaction_id = source.transaction_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"Merged {incremental_count} rows into {bronze_table_name}")
else:
    print("No new rows to merge")

In [ ]:
# Update the watermark only when new rows were processed.
if incremental_count > 0:
    new_watermark = incremental_df.agg(F.max("transaction_timestamp").alias("max_ts")).collect()[0]["max_ts"]
    watermark_update = spark.createDataFrame(
        [("transactions", "transaction_timestamp", str(new_watermark), datetime.now(timezone.utc))],
        ["entity_name", "watermark_column", "watermark_value", "updated_at"]
    )
    watermark_update.write.format("delta").mode("append").saveAsTable(watermark_table_name)
    print(f"Updated watermark to {new_watermark}")
else:
    print("Watermark unchanged")

display(spark.table(watermark_table_name).orderBy(F.col("updated_at").desc()))

## Production Considerations

- Store one watermark per source entity and environment.
- Reconcile source and target row counts for each batch.
- Decide whether late-arriving records should use event time or extraction time.
- Keep a replay strategy for failed batches.
- Use pipeline parameters for source path, entity name, and batch ID.